In [3]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()


True

In [ ]:
simple_worker = create_agent(
    model="gemini-3.5-flash",
    tools=[search, send_push_notifications, wikipedia_lookup]
    system_prompts = "You are a Sidekick, a helpful personal assistant. Use your tools to complete the task.",
    checkpointer=InMemorySaver()
)

In [ ]:
async def ask(worker, message):
    config = {"configurable": {"thread_id": "simple-sidekick"}}
    result = await worker.ainvoke({"messages": [{"role": "user", "content": message}]}, config=config)
    return result["messages"][-1].text
reply = await ask(simple_worker, "Search for who won the Nobel Prize in Physics in 2023 and send a push notification with a short summary.")
print(reply)

In [ ]:
# human in the loop middleware tool
from langchain_core.tools import tool

@tool
def book_meeting(person: str, day:str) -> str:
    """Book a meeting with a person on a given day."""
    return f"Metting booked with {person} on {day}"

approval_agent = create_agent(
    model="gemini-flash-3.5",
    tools=[book_meeting],
    system_prompt="You are a scheduling assistant. Use the book_meeting tool.",
    middleware=[HumanInTheLoopMiddleware[AgentState[any], None, any](interrupt_on={"book_meeting": True})],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "approval-demo"}}
result = await approval_agent.ainvoke(
    {"messages": [{"role": "user", "content": "Book a meeting with sam on Friday"}]}, config
)

interrupt = result["__interrupt__"][0]
print("The agent paused and is asking for approval:")
print(interrupt.value["action_requests"][0]["description"])

In [ ]:
resumed = await approval_agent.ainvoke(Command(sresume={"decisions": [{"type": "approve"}]}, config))
print(resumed["messages"][-1].content)

In [ ]:
sidekick = Sidekick()
await sidekick.setup()
print(f"Sidekick ready with {len(sidekick.tools)} tools")

In [ ]:
history = await sidekick.run_turn(
    message="Go to Hacker News at new.ycombinator.com and tell me the title of the current top story.",
    success_criteria="The reply names a specific story currently on the Hacker New front page.",
)
for entry in history:
    print(f"[(entry['role])] {entry['content'][:200]}\n")

In [4]:
flight_task = """Find me the best round-trip flight from New York to London, leaving about a month from now
and returning a week later. I care about price first, then total journey time, and I would rather avoid
iteneraries with two or more stops. Write your recommendation with the top three options to flights.md,
then send me a push notification with the price of your top pick.
"""

flight_criteria = "flights.md is written with three specific options including airline, times and price, plus a clear recommendations, and send me a push notification with your top pick "

history = await sidekick.run_turn(flight_task, flight_criteria, history)
print(history[-1]["content"])

NameError: name 'sidekick' is not defined

In [ ]:
for todo in sidekick.todos:
    print(f"[{todo['status']}] {todo['content']}")